In [1]:
import requests
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime

# Project Root
ROOT = Path.cwd().parent

# Load Model and Encoders
model = joblib.load(ROOT / "models" / "weather_model.pkl")
le_city = joblib.load(ROOT / "models" / "le_city.pkl")
le_weather = joblib.load(ROOT / "models" / "le_weather.pkl")
le_day = joblib.load(ROOT / "models" / "le_day.pkl")
le_season = joblib.load(ROOT / "models" / "le_season.pkl")

print("✅ Model & Encoders loaded successfully!")


✅ Model & Encoders loaded successfully!


In [2]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

API_KEY = "1becb0fb56249486d02f3d4f89203315"
city = "Mangalore"
url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

response = requests.get(url)
if response.status_code == 200:
    weather_data = response.json()
    temp_current = weather_data["main"]["temp"]
    humidity = weather_data["main"]["humidity"]
    pressure = weather_data["main"]["pressure"]
    wind_speed = weather_data["wind"]["speed"]
    cloud_cover = weather_data["clouds"]["all"]
    rainfall = weather_data.get("rain", {}).get("1h", 0.0)
    weather_cond = weather_data["weather"][0]["main"]
    
    now = datetime.now()
    year, month, day, hour = now.year, now.month, now.day, now.hour
    day_name = now.strftime("%A")
    season = get_season(month)
    
    # Encode Categoricals
    city_enc = le_city.transform([city])[0] if city in le_city.classes_ else 0
    weather_enc = le_weather.transform([weather_cond])[0] if weather_cond in le_weather.classes_ else 0
    day_enc = le_day.transform([day_name])[0] if day_name in le_day.classes_ else 0
    season_enc = le_season.transform([season])[0] if season in le_season.classes_ else 0
    
    input_features = pd.DataFrame([{
        "City": city_enc,
        "Humidity": humidity,
        "Pressure": pressure,
        "Wind_Speed": wind_speed,
        "Cloud_Cover": cloud_cover,
        "Rainfall": rainfall,
        "Weather": weather_enc,
        "Year": year,
        "Month": month,
        "Day": day,
        "Hour": hour,
        "DayOfWeek": day_enc,
        "Season": season_enc
    }])
    
    predicted_temp = model.predict(input_features)[0]
    print(f"Live Weather in {city}: {weather_cond}, Current Temp: {temp_current}°C")
    print(f"Predicted Temperature: {predicted_temp:.2f}°C")
else:
    print("API request failed with status:", response.status_code)


ValueError: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.
